In [1]:
import numpy as np

In [60]:
def create_include(fname):
    with open(fname, 'a') as main:
        main.write("#include <core.p4>\n")
        main.write("#include <tna.p4>\n\n")

def create_main(fname):
    with open(fname, 'a') as main:
        main.write("/*************************************************************************\n"
                     "***********************  S W I T C H  ************************************\n"
                     "*************************************************************************/\n\n"
                     "Pipeline(SwitchIngressParser(),\n"
                     "    SwitchIngress(),\n"
                     "    SwitchIngressDeparser(),\n"
                     "    SwitchEgressParser(),\n"
                     "    SwitchEgress(),\n"
                     "    SwitchEgressDeparser()) pipe;\n\n"
                     "Switch(pipe) main;")

def write_tables(fname, config):
    with open(fname, 'a') as ingress:

        if config['input_layer'] == True:
            match_type = "range"
        else:
            match_type = "exact"
        for f in range(0, config['num_features']):
            ingress.write("    table lookup_feature" + str(f) + " {\n"
                     "        key = { hdr.Planter_in.feature" + str(f) + ":" + match_type + "; }\n"
                     "        actions = {\n"
                     "            extract_feature" + str(f) + "();\n"
                     "            NoAction;\n"
                     "            }\n"
                     "        size = " + str(256) + ";\n"
                     "        default_action = NoAction;\n"
                     "    }\n\n")

def get_ranges(num_features, divider):
    ranges = []

    if num_features % 2 != 0:
        num = num_features + 1
    else:
        num = num_features

    for i in range(0, num, int(num/divider)):
        end = min(i + int(num/divider), num)
        ranges.append(list(range(i, end)))
    return ranges

def write_actions(fname, config):
    index = 0
    with open(fname, 'a') as ingress:

        for f in range(0, config['num_features']):
            ranges = get_ranges(config['num_features'], config['num_tem'])
            for i in range(config['num_tem']):
                if f in ranges[i]:
                    index = i
                    break
            ingress.write("    action extract_feature" + str(f)+'(')
            for h in range(0, config['num_nodes']):
                if h==0:
                    ingress.write("int<" + str(config['num_bits']) + "> w_" + str(f) + "_" + str(h))
                else:
                    ingress.write(", int<" + str(config['num_bits']) + "> w_" + str(f) + "_" + str(h))
            ingress.write("){\n")
            for h in range(0, config['num_nodes']):
                ingress.write("        meta.tem"+ str(index) + "_h" + str(h)+" = meta.tem"+ str(index) + "_h" + str(h)
                              + " + w_" + str(f)+ "_" + str(h) + ";\n")
            ingress.write("    }\n\n")

def create_ingress_control(fname, config):
    with open(fname, 'a') as ingress:
        ingress.write("/*************************************************************************\n"
                      "*********************** Ingress Processing********************************\n"
                      "**************************************************************************/\n\n")

        ingress.write("control SwitchIngress(\n    inout header_t hdr,\n"
                      "    inout metadata_t meta,\n"
                      "    in ingress_intrinsic_metadata_t ig_intr_md,\n"
                      "    in ingress_intrinsic_metadata_from_parser_t ig_prsr_md,\n"
                      "    inout ingress_intrinsic_metadata_for_deparser_t ig_dprsr_md,\n"
                      "    inout ingress_intrinsic_metadata_for_tm_t ig_tm_md) {\n\n")

        ingress.write("    action drop() {\n"
                      "        ig_dprsr_md.drop_ctl = 0x1;\n"
                      "    }\n\n" )

        ingress.write("    action send(PortId_t port) {\n"
                      "        ig_tm_md.ucast_egress_port = port;\n"
                      "    }\n\n")

    write_actions(p4_file, config)
    write_tables(p4_file, config)

    with open(fname, 'a') as ingress:
        ingress.write("    apply{\n")
        # ingress.write("        bias.apply();\n")
        for f in range(0, config['num_features']):
            ingress.write("        lookup_feature" + str(f) + ".apply();\n")

        if config['num_tem'] == 4:
        # add 0 and 1
            ingress.write("\n")
            for f in range(0, config['num_nodes']):
                ingress.write("        meta.tem0_h" + str(f) + " =")
                ingress.write(" meta.tem0_h" + str(f) + " + meta.tem1_h" + str(f) +";\n")
        # add 2 and 3
            ingress.write("\n")
            for f in range(0, config['num_nodes']):
                ingress.write("        meta.tem2_h" + str(f) + " =")
                ingress.write(" meta.tem2_h" + str(f) + " + meta.tem3_h" + str(f) +";\n")

        # add the last two tem values
        ingress.write("\n")
        ingress.write("        //add the last two temp value\n")
        for f in range(0, config['num_nodes']):
            ingress.write("        meta.tem0_h" + str(f) + " =")
            if config['num_tem'] == 4:
                ingress.write(" meta.tem0_h" + str(f) + " + meta.tem2_h" + str(f) +";\n")
            else:
                ingress.write(" meta.tem0_h" + str(f) + " + meta.tem1_h" + str(f) +";\n")

        # write the clamping function
        ingress.write("\n")
        if config['output_layer'] != True:
          ingress.write("        //clamping function\n")
          for i in range(0, config['num_nodes']):
              ingress.write("        if ((meta.tem0_h" + str(i) + " & ~32s0xFFFFFF80) != 0 && meta.tem0_h" + str(i) + "[31:31]==1) {meta.tem0_h" + str(i) + " = -128; }\n")
              ingress.write("        else if ((meta.tem0_h" + str(i) + " & 32s0xFFFFFF80) != 0 && meta.tem0_h" + str(i) + "[31:31]==0) {meta.tem0_h" + str(i) + " = 127; }\n")

        if config['num_features'] != config['num_nodes'] and config['output_layer'] != True:
            ingress.write("\n")
            ingress.write("        hdr.Planter_out.setValid();\n")
            ingress.write("        hdr.Planter_in.setInvalid();\n")
            ingress.write("\n")

        # give value for the out header
        ingress.write("\n")
        if config['output_layer'] == True:
            ingress.write("        //give labels\n")
            for i in range(0, config['num_nodes']):
                ingress.write("        if(meta.tem0_h" + str(i) + " < 0) {hdr.Planter_in.feature" + str(i) + " = 0; }\n")
                ingress.write("        else {hdr.Planter_in.feature" + str(i) + " = 1; }\n")
        elif config['num_features'] == config['num_nodes']:
            ingress.write("        //give value for the out header\n")
            for i in range(0, config['num_nodes']):
                ingress.write("        hdr.Planter_in.feature" + str(i) + " = meta.tem0_h" + str(i) + " + one_two_eight;\n")
        else:
            ingress.write("        //give value for the out header\n")
            for i in range(0, config['num_nodes']):
                ingress.write("        hdr.Planter_out.node" + str(i) + " = meta.tem0_h" + str(i) + " + one_two_eight;\n")

        ingress.write("\n")
        ingress.write("        send(ig_intr_md.ingress_port);\n")
        ingress.write("    }\n" )
        ingress.write("}\n")

def create_headers(fname, config):
     with open(fname, 'a') as headers:
          headers.write("const bit<16> ETHERTYPE_Planter = 0x1234;\n")
          headers.write("const int<32> one_two_eight = 128;\n\n")
          headers.write("header ethernet_h {\n"
                      "    bit<48> dstAddr;\n"
                      "    bit<48> srcAddr;\n"
                      "    bit<16> etherType;\n"
                      "}\n\n" )
          # write the Planter_in header
          headers.write("header Planter_in_h{\n")
          if config['input_layer'] == True:
              for f in range(0, config['num_features']):
                  headers.write("    bit<16> feature" + str(f) + ";\n")
          else:
              for f in range(0, config['num_features']):
                  headers.write("    int<32> feature" + str(f) + ";\n")

          headers.write("}\n\n")
          # write the Planter_out header
          if config['num_features'] != config['num_nodes'] and config['output_layer'] != True:
              headers.write("header Planter_out_h{\n")
              for f in range(0, config['num_nodes']):
                  headers.write("    int<32> node" + str(f) + ";\n")
              headers.write("}\n\n")

          # write the headers' structure
          headers.write("struct header_t {\n"
                      "    ethernet_h   ethernet;\n"
                      "    Planter_in_h    Planter_in;\n")

          if config['num_features'] != config['num_nodes'] and config['output_layer'] != True:
              headers.write(
                      "    Planter_out_h    Planter_out;\n"
                      "}\n\n")
          else:
              headers.write("}\n\n")

def create_metadata(fname, config):
     with open(fname, 'a') as metadata:
        metadata.write("struct metadata_t{\n")
        for j in range(0, config['num_tem']):
            for i in range(0, config['num_nodes']):
                metadata.write("    int<" + str(config['num_bits']) + "> tem" + str(j) + "_h" + str(i) + ";\n")
        metadata.write("}\n\n")

def create_parser(fname, config):
    with open(fname, 'a') as parser:
        parser.write("/*************************************************************************\n"
                     "*********************** Ingress Parser ***********************************\n"
                     "*************************************************************************/\n\n")

        parser.write(
            "parser SwitchIngressParser(\n"
            "    packet_in pkt,\n"
            "    out header_t hdr,\n"
            "    out metadata_t meta,\n"
            "    out ingress_intrinsic_metadata_t ig_intr_md) {\n\n")

        parser.write("    state start {\n"
                     "        pkt.extract(ig_intr_md);\n"
                     "        pkt.advance(PORT_METADATA_SIZE);\n"
                     "        transition parse_ethernet;\n"
                     "    }\n\n")

        parser.write("    state parse_ethernet {\n"
                     "        pkt.extract(hdr.ethernet);\n"
                     "        transition select(hdr.ethernet.etherType) {\n"
                     "        ETHERTYPE_Planter : parse_planter;\n"
                     "        default           : accept;\n"
                     "        }\n"
                     "    }\n\n" )

        parser.write("    state parse_planter {\n"
                     "        pkt.extract(hdr.Planter_in);\n"
                     "        transition accept;\n"
                     "    }\n")

    with open(fname, 'a') as parser:
        parser.write("}\n\n")
        parser.write("/*************************************************************************\n"
                     "*********************** Ingress Deparser *********************************\n"
                     "**************************************************************************/\n\n")

        parser.write("control SwitchIngressDeparser(\n"
                     "    packet_out pkt,\n"
                     "    inout header_t hdr,\n"
                     "    in metadata_t ig_md,\n"
                     "    in ingress_intrinsic_metadata_for_deparser_t ig_dprsr_md) {\n"
                     "    apply {\n")
        # change the header to the out header

        if config['num_features'] != config['num_nodes'] and config['output_layer'] != True:
            parser.write(
                     "        pkt.emit(hdr.ethernet);\n"
                     "        pkt.emit(hdr.Planter_out);\n"
                     "    }\n}\n\n")

        else:
            parser.write(
                     "        pkt.emit(hdr);\n"
                     "    }\n}\n\n")


        parser.write("/*************************************************************************\n"
                     "*********************** Egress Parser ***********************************\n"
                     "*************************************************************************/\n\n")

        parser.write(
                      "parser SwitchEgressParser(\n"
                      "    packet_in pkt,\n"
                      "    out header_t hdr,\n"
                      "    out metadata_t meta,\n"
                      "    out egress_intrinsic_metadata_t eg_intr_md) {\n")

        parser.write("    state start {\n"
                     "        pkt.extract(eg_intr_md);\n"
                     "        transition accept;\n"
                     "    }\n"
                     "}\n\n")

        parser.write("/*************************************************************************\n"
                     "*********************** Egress Deparser *********************************\n"
                     "**************************************************************************/\n\n")

        parser.write("control SwitchEgressDeparser(\n"
                     "    packet_out pkt,\n"
                     "    inout header_t hdr,\n"
                     "    in metadata_t eg_md,\n"
                     "    in egress_intrinsic_metadata_for_deparser_t eg_dprsr_md) {\n"
                     "    apply {\n"
                     "        pkt.emit(hdr);\n"
                     "    }\n"
                     "}\n\n")

def create_egress_control(fname, config):
    with open(fname, 'a') as egress:
        egress.write("/*************************************************************************\n"
                     "*********************** Egress Processing********************************\n"
                     "**************************************************************************/\n\n"
                     "control SwitchEgress(inout header_t hdr,\n"
                     "    inout metadata_t meta,\n"
                     "    in egress_intrinsic_metadata_t eg_intr_md,\n"
                     "    in egress_intrinsic_metadata_from_parser_t eg_prsr_md,\n"
                     "    inout egress_intrinsic_metadata_for_deparser_t     eg_dprsr_md,\n"
                     "    inout egress_intrinsic_metadata_for_output_port_t  eg_oport_md) {\n\n"
                     "    action drop() {\n"
                     "        eg_dprsr_md.drop_ctl = 0x1;\n"
                     "    }\n\n"
                     )

        # The following implements static routing between two ports, change as necessary
        egress.write("    apply {\n")
        egress.write("    }\n}\n")

In [61]:
p4_file = 'MTL_layer3' + '.p4'

config={}
config['num_features'] = 8
config['num_nodes'] = 4

config["input_layer"] = False
config["output_layer"] = True

# action bits
config['num_bits'] = 32

# parallel execution
config['num_tem'] = 4

create_include(p4_file)

create_headers(p4_file, config)

create_metadata(p4_file, config)

create_parser(p4_file, config)

create_ingress_control(p4_file, config)

create_egress_control(p4_file, config)

create_main(p4_file)